# Genizah v2.3a — first Arabic-script run

Same training stack as v2.2b (pinned install triplet, 6.5/7 MP resolution contract, LoRA r16 on tower +
language, merger FROZEN, adamw_8bit, cosine, warm start = **v2.1b step 1200**, fixed row order with exact
resume, a session stops itself after 21.5 h with a save and a Hub push). What changes:

| | v2.2b | this run |
|---|---|---|
| Data | `isaacmg/genizah_v22_full` | `isaacmg/genizah_v23a_arabic`: Arabic-script pages (Geniza editions, Agapet, Muharaf, BAYBARS, ISKANDAR) + Hebrew-script replay (full documentary pages, KTIV transcription) |
| Rows / steps | 20,000 / 2,500 | 12,000 / 1,500 |
| Learning rate | 3e-5, 2 % warm-up | 1e-4, 5% warm-up: a new script is learned, not adjusted |
| Gates | boxes, QA | Arabic script in Arabic sources, Hebrew script in replay |

**To resume after a session ends: run every cell again, unedited.** The notebook finds the newest checkpoint on the
Hub, checks the stored schedule and row order, and continues with the rows not yet trained.

Run name `genizah_v23a`; checkpoints in `isaacmg/qwen3-vl-8b-hebrew-v23a-ckpt`. Checkpoints are scored on the Arabic-script benchmark
(`docs/arabic_script_benchmark.md`) and on the Hebrew-script benchmarks for regression.


In [ ]:
# Cell 1 — installs + env (PINNED: the exact triplet audited 2026-08-09;
# unpinned installs float and transformers 5.x would invalidate the vision-path
# analysis AND the 108-tensor count)
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["WANDB_PROJECT"] = "qwen-hebrew-finetune"
%pip install -q "unsloth[colab-new]==2026.8.9" "unsloth_zoo==2026.8.6" "transformers==4.57.6" hf_transfer wandb

from importlib.metadata import version
assert version("unsloth_zoo") == "2026.8.6", f"unsloth_zoo drifted: {version('unsloth_zoo')}"
assert version("transformers") == "4.57.6", f"transformers drifted: {version('transformers')}"

import torch
assert torch.cuda.is_available(), "No GPU — switch runtime to A100"

In [ ]:
# Cell 2 — data: genizah_v22_pilot (images-once, MAP-STYLE, materialised mixture — no streaming)
import json
from collections import Counter
from pathlib import Path
from google.colab import userdata
from huggingface_hub import HfApi, login, snapshot_download

login(token=userdata.get("HF_TOKEN"))

V22_REPO = "isaacmg/genizah_v23a_arabic"
V22_REVISION = "3b762524067c77141c121ce84a3a724991618757"   # v23a Arabic mixture: rows + images_part*.tar (12,000 train rows)
if V22_REVISION == "PIN-AFTER-PUSH":
    V22_REVISION = HfApi().dataset_info(V22_REPO).sha
    print(f"⚠️ unpinned: {V22_REPO} main -> {V22_REVISION} — pin this SHA before the next launch")
assert len(V22_REVISION) == 40, f"unpinned revision: {V22_REVISION!r}"
# images ship as two archives (images_part1.tar, images_part2.tar): 4,598 separate JPEGs took >45 min through the per-file path on 2026-09-25
LOCAL = "/content/genizah_v23a_arabic"
DATA_DIR = Path(snapshot_download(V22_REPO, repo_type="dataset", revision=V22_REVISION, local_dir=LOCAL,
                                  allow_patterns=["rows/*", "*.json", "README.md", "images*.tar"]))
_manifest = json.loads((DATA_DIR / "manifest.json").read_text())
_expected = sum(s["unique_images"] for s in _manifest["splits"].values())
_have = len(list((DATA_DIR / "images").glob("*.jpg"))) if (DATA_DIR / "images").is_dir() else 0
if _have < _expected:
    _tars = sorted(DATA_DIR.glob("images*.tar"))
    if _tars:
        import tarfile
        for _t in _tars:
            with tarfile.open(_t) as _tf:
                try:
                    _tf.extractall(DATA_DIR, filter="data")
                except TypeError:                  # Python < 3.12: no filter argument
                    _tf.extractall(DATA_DIR)
            _t.unlink()
    else:                                          # revision without the archive: per-file fallback (slow)
        snapshot_download(V22_REPO, repo_type="dataset", revision=V22_REVISION, local_dir=LOCAL, allow_patterns=["images/*"])
    _have = len(list((DATA_DIR / "images").glob("*.jpg")))
assert _have >= _expected, f"images: {_have} present, {_expected} expected"
print(f"images ready: {_have}")
MIX = json.loads((DATA_DIR / "mixture.json").read_text())
print("mixture.json:", json.dumps(MIX, ensure_ascii=False)[:800])

import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from datasets import Image as _HFImage
from torch.utils.data import Dataset as _TorchDataset

META_KEY = b"images_once"
SHA_COLUMN = "image_sha1"


class ImagesOnceDataset(_TorchDataset):
    """Map-style rows with images stored once (mirror of src/finetuning/qwen_hebrew/images_once.py).

    Items equal the original DatasetDict rows (image decoded, columns in the original order), so the
    v2.1b collator works unchanged. Rows stay in one Arrow table: forked DataLoader workers share it.
    """

    def __init__(self, rows_parquet, images_dir, tasks=None, check_files=True):
        table = pq.read_table(rows_parquet)
        meta = json.loads(table.schema.metadata[META_KEY])
        if tasks is not None:
            table = table.filter(pc.is_in(table["task"], value_set=pa.array(list(tasks))))
        self.images_dir = Path(images_dir)
        self.columns = meta["columns"]
        self.image_column = meta["image_column"]
        self._decoder = _HFImage(mode=meta["image_mode"])
        self._table = table.combine_chunks()
        if check_files:
            missing = [s for s in pc.unique(table[SHA_COLUMN]).to_pylist()
                       if not self.image_path(s).exists()]
            assert not missing, f"{len(missing)} images missing under {images_dir}, e.g. {missing[0]}.jpg"

    def __len__(self):
        return self._table.num_rows

    def image_path(self, sha1):
        return self.images_dir / f"{sha1}.jpg"

    def load_image(self, sha1):
        return self._decoder.decode_example({"path": None, "bytes": self.image_path(sha1).read_bytes()})

    def column(self, name):
        return self._table[name].to_pylist()

    def __getitem__(self, idx):
        n = len(self)
        if not -n <= idx < n:
            raise IndexError(f"row {idx} out of range for {n} rows")
        row = self._table.slice(idx % n, 1).to_pylist()[0]
        return {c: self.load_image(row[SHA_COLUMN]) if c == self.image_column else row[c]
                for c in self.columns}


train_ds = ImagesOnceDataset(DATA_DIR / "rows/train.parquet", DATA_DIR / "images")
eval_ds = ImagesOnceDataset(DATA_DIR / "rows/val.parquet", DATA_DIR / "images")
print(f"train rows={len(train_ds)} val rows={len(eval_ds)}")
assert len(train_ds) >= 11400 and len(eval_ds) >= 100, "Arabic mixture incomplete"

_src = Counter(train_ds.column("source"))
_task = Counter(train_ds.column("task"))
_n = len(train_ds)
print("train by source:", {k: (v, round(v / _n, 3)) for k, v in _src.items()})
print("train by task:", dict(_task))
for _s, _lo in (("arabic_editions", 0.09), ("arabic_agapet", 0.06), ("arabic_muharaf", 0.18), ("arabic_baybars", 0.07), ("arabic_iskandar", 0.02), ("pgp_edition_pages", 0.15), ("ktiv_transcription", 0.22)):
    assert _src[_s] / _n >= _lo, f"{_s} share {_src[_s] / _n:.3f} below {_lo} — wrong dataset build"
assert set(_src) <= set(eval_ds.column("source")), "val must cover every train source"

# Label hygiene (transcription answers) — same checks as v2.0a/v2.1b
_ans = train_ds.column("answer")
_q = train_ds.column("question")
_t = train_ds.column("task")
_srcs = train_ds.column("source")
_transc = [a for a, t in zip(_ans, _t) if t.endswith("transcribe")]
assert _transc and all("␣" not in a for a in _transc), "internal gap token leaked"
assert all(not any(ch in a for ch in "&#$_{}<>\\") for a in _transc), "mojibake leaked"
assert any("[...]" in a for a in _transc), "expected damage-gap markers"

# Arabic hygiene: an Arabic source's answer is mainly Arabic script, its prompt names the script, outside targets
# carry no vowel marks; the replay rows stay Hebrew script. This mixture has no box rows and no QA rows.
import re as _re
_AR, _HE = _re.compile(r"[\u0620-\u064a]"), _re.compile(r"[\u05d0-\u05ea]")
_MARKS = _re.compile(r"[\u0640\u064b-\u0652]")
_arabic = [(a, q, s) for a, q, s in zip(_ans, _q, _srcs) if s.startswith("arabic_")]
assert _arabic, "no Arabic-script rows in the mixture"
for a, q, s in _arabic:
    _n_ar, _n_he = len(_AR.findall(a)), len(_HE.findall(a))
    assert _n_ar >= 40 and _n_ar >= 9 * _n_he, f"{s}: answer is not mainly Arabic script ({_n_ar} Arabic / {_n_he} Hebrew letters)"
    assert "Arabic script" in q, f"{s}: the prompt does not name Arabic script"
assert all(not _MARKS.search(a) for a, q, s in _arabic if s != "arabic_editions"), "vowel marks or tatweel left in an outside target"
_replay = [a for a, s in zip(_ans, _srcs) if not s.startswith("arabic_")]
assert _replay and sum(len(_HE.findall(a)) for a in _replay) > 20 * sum(len(_AR.findall(a)) for a in _replay), \
    "the replay rows are not Hebrew script"
assert not any(t in ("locate", "locate_word", "read_box", "read_box_word") or s == "pgp_qa" for t, s in zip(_t, _srcs)), \
    "box or QA rows in an Arabic mixture"
print(f"hygiene OK: transcription {len(_transc)}, Arabic rows {len(_arabic)} "
      f"({Counter(s for *_, s in _arabic)}), Hebrew replay rows {len(_replay)}")


In [ ]:
# Cell 3 — model at page resolution + THE MERGER KNOB + flagship warm start (v2.1b step 1200)
from unsloth import FastVisionModel
from unsloth_zoo.peft_utils import get_peft_regex
from transformers import AutoImageProcessor

MAX_SEQ = 12288
# Global 6.5MP floor (train/infer resolution contract, ships in the export).
MIN_PIX = 6_500_000
MAX_PIX = 7_000_000

# ⬅️ THE DECISION KNOB — set per the v1.9c verdict:
#   "frozen" = v1.9a config (default, safe)   "full" = v1.9c full-weight merger
#   "lora"   = v1.9b merger LoRA (two-sided at r16 per the ablation; avoid)
MERGER_MODE = "frozen"
assert MERGER_MODE in ("frozen", "lora", "full")

# Warm start: v2.1b step 1200 — the flagship (religious CER 0.167, PGP 0.180, boxes on the right line 80%;
# the 1300–1700 checkpoints were never promoted). The v2.2 data lever sits on top of it.
WARM_CKPT_REPO = "isaacmg/qwen3-vl-8b-hebrew-v21b-ckpt"
WARM_REVISION = "6724c32cd0f6296223978ae69635fc782d2dbb03"  # commit "Training in progress, step 1200, checkpoint"

TRAIN_VISION_LORA = True    # unchanged since v1.8b
MERGER_MODULES = [
    "visual.merger.linear_fc1", "visual.merger.linear_fc2",
    *[f"visual.deepstack_merger_list.{i}.linear_fc{j}"
      for i in range(3) for j in (1, 2)],
]

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3-VL-8B-Instruct",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
    max_seq_length=MAX_SEQ,
)

base_regex = get_peft_regex(
    model,
    finetune_vision_layers=True, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
)
MERGER_REGEX = r".*\.visual\.(?:merger|deepstack_merger_list\.\d+)\.linear_fc[12]"
target_modules = (f"(?:{base_regex})|(?:{MERGER_REGEX})"
                  if MERGER_MODE == "lora" else base_regex)

if MERGER_MODE == "full":
    # modules_to_save cannot train a quantized clone (see v1.9c notebook)
    import bitsandbytes as bnb
    _mods = {n: m for n, m in model.named_modules()
             if any(n.endswith(s) for s in MERGER_MODULES)}
    assert len(_mods) == 8, f"merger modules found: {sorted(_mods)}"
    _quantized = [n for n, m in _mods.items()
                  if isinstance(m, (bnb.nn.Linear4bit, bnb.nn.Linear8bitLt))]
    assert not _quantized, (
        f"merger modules are quantized: {_quantized} — reload with these in "
        "llm_int8_skip_modules or load_in_4bit=False before proceeding.")

model = FastVisionModel.get_peft_model(
    model,
    target_modules=target_modules,
    modules_to_save=MERGER_MODULES if MERGER_MODE == "full" else None,
    finetune_vision_layers=True, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=16, lora_alpha=16, lora_dropout=0.0, bias="none", random_state=3407,
)

# creation gate — the run is void unless the adapter matches the KNOB
_tower_lora = [n for n, _ in model.named_parameters()
               if ".visual.blocks." in n and "lora_A" in n]
_merger_lora = [n for n, _ in model.named_parameters()
                if "merger" in n and "lora_A" in n]
_merger_full = [n for n, p in model.named_parameters()
                if "merger" in n and ".modules_to_save." in n
                and n.endswith(".weight") and p.requires_grad]
assert len(_tower_lora) == 108, f"tower adapters: {len(_tower_lora)}/108"
if MERGER_MODE == "frozen":
    assert not _merger_lora and not _merger_full, "merger adapted but knob=frozen"
elif MERGER_MODE == "lora":
    assert len(_merger_lora) == 8 and not _merger_full,         f"merger lora {len(_merger_lora)}/8 — knob=lora not honoured"
else:
    assert len(_merger_full) == 8 and not _merger_lora,         f"merger clones {len(_merger_full)}/8 — knob=full not honoured"
print(f"MERGER_MODE={MERGER_MODE}: tower {len(_tower_lora)}/108, "
      f"merger lora {len(_merger_lora)}, merger full {len(_merger_full)}")

# weights-only warm start (fresh optimizer + schedule)
assert len(WARM_REVISION) == 40, "invalid revision SHA"
from safetensors.torch import load_file
from peft import set_peft_model_state_dict
local = snapshot_download(WARM_CKPT_REPO, revision=WARM_REVISION,
                          allow_patterns="last-checkpoint/adapter_model.safetensors")
_sd = load_file(f"{local}/last-checkpoint/adapter_model.safetensors")
_loaded_merger_lora = any("merger" in k and "lora" in k for k in _sd)
_loaded_merger_full = any("merger" in k and "lora" not in k for k in _sd)
# (modules_to_save saves PLAIN keys — the adapter-name infix is stripped on save)
# the knob must match what the warm checkpoint carries, or trained merger
# weights would be silently dropped on load
if _loaded_merger_full:
    assert MERGER_MODE == "full", "warm ckpt has full merger weights — set MERGER_MODE='full'"
if _loaded_merger_lora:
    assert MERGER_MODE == "lora", "warm ckpt has merger LoRA — set MERGER_MODE='lora'"
if MERGER_MODE == "full":
    # PEFT's modules_to_save loader KeyErrors when the incoming dict lacks the
    # merger keys (any pre-v1.9c warm start). Inject the base weights — the
    # identity start the gate below verifies; setdefault keeps warm-loaded
    # merger weights when the checkpoint carries them.
    for _n, _m in model.named_modules():
        if any(_n.endswith(s) for s in MERGER_MODULES) and hasattr(_m, "modules_to_save"):
            for _pn, _p in _m.original_module.named_parameters():
                _sd.setdefault(f"{_n}.{_pn}", _p.detach().clone())
missing = set_peft_model_state_dict(model, _sd)
print("unexpected keys:", len(getattr(missing, "unexpected_keys", [])))
_wv = [n for n, p in model.named_parameters()
       if ".visual.blocks." in n and "lora_B" in n and p.detach().abs().max().item() > 0]
assert len(_wv) == 108, f"warm start vision adapter not loaded: {len(_wv)}/108 nonzero"
if MERGER_MODE == "lora" and not _loaded_merger_lora:
    _mz = [n for n, p in model.named_parameters()
           if "merger" in n and "lora_B" in n and p.detach().abs().max().item() > 0]
    assert not _mz, f"merger lora_B nonzero at start: {_mz[:3]}"
if MERGER_MODE == "full":
    _wrappers = [(n, m) for n, m in model.named_modules()
                 if any(n.endswith(s) for s in MERGER_MODULES)
                 and hasattr(m, "modules_to_save")]
    assert len(_wrappers) == 8, f"modules_to_save wrappers: {len(_wrappers)}/8"
    if not _loaded_merger_full:
        for _n, _w in _wrappers:
            assert torch.allclose(
                _w.modules_to_save["default"].weight.detach().float(),
                _w.original_module.weight.detach().float()),                 f"merger clone differs from base at start: {_n}"
        print("merger clones at identity (fresh full-weight start)")
    else:
        print("merger clones warm-loaded from checkpoint")
print(f"warm start OK from {WARM_CKPT_REPO}@{WARM_REVISION[:8]}")

if TRAIN_VISION_LORA:
    # patch-embed require-grad fix (see v1.8b/v1.9 notebooks for the diagnosis)
    def _vision_embeds_require_grad(module, inputs, output):
        if not torch.is_grad_enabled():
            return output
        output.requires_grad_(True)
        return output
    _patch_embed = next(m for n, m in model.named_modules()
                        if n.endswith("visual.patch_embed"))
    _patch_embed.register_forward_hook(_vision_embeds_require_grad)
    print("vision LoRA gradient fix: ON")

tokenizer.image_processor = AutoImageProcessor.from_pretrained(
    "unsloth/Qwen3-VL-8B-Instruct", min_pixels=MIN_PIX, max_pixels=MAX_PIX,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable: {trainable/1e6:.1f}M")
print("resolution:", tokenizer.image_processor.size)


In [ ]:
# Cell 4 — conversation format + collator (native res preserved by resize='max')
def to_conversation(sample):
    return {
        "messages": [
            {"role": "user", "content": [
                {"type": "image", "image": sample["image"]},
                {"type": "text", "text": sample["question"]},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": sample["answer"]},
            ]},
        ]
    }

from unsloth.trainer import UnslothVisionDataCollator

collator = UnslothVisionDataCollator(
    model, tokenizer,
    formatting_func=to_conversation,
    resize="max",
    max_seq_length=MAX_SEQ,
    train_on_responses_only=True,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# guardrail: one real batch (a KTIV page + an Arabic-script page) must show page-res pixels + masked labels
_i_page = next(i for i, (t, s) in enumerate(zip(train_ds.column("task"), train_ds.column("source")))
               if t == "fragment_transcribe" and s.startswith("ktiv"))
_i_qa = next(i for i, s in enumerate(train_ds.column("source")) if s.startswith("arabic_"))   # the Arabic page
_page_row, _qa_row = train_ds[_i_page], train_ds[_i_qa]
batch = collator([_page_row, _qa_row])
pv = batch["pixel_values"]
assert pv is not None and pv.shape[0] > 40000, (
    f"{pv.shape[0]} patch rows — expected >40k for two ~6.5MP images; "
    "the resolution policy is not reaching the collator")
labels = batch["labels"]
_tok = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer
for _k, _row in ((0, _page_row), (1, _qa_row)):
    _unmasked = labels[_k][labels[_k] != -100]
    assert _row["answer"][:30] in _tok.decode(_unmasked), f"row {_k}: answer not in the unmasked labels"
    assert _row["question"][:30] not in _tok.decode(_unmasked), f"row {_k}: prompt leaked into the loss"
print(f"collator OK: pixel rows={pv.shape[0]}, labels masked on both rows (Hebrew page + Arabic page)")


In [ ]:
# Cell 5 — gradient-flow verification (cheap; run BEFORE burning GPU-hours)
# Language LoRA always; tower LoRA iff TRAIN_VISION_LORA; merger per the KNOB.
FastVisionModel.for_training(model)
_vb = {k: (v.to(model.device) if torch.is_tensor(v) else v)
       for k, v in collator([train_ds[_i_page]]).items()}
model(**_vb).loss.backward()
tower_g = [p.grad.abs().max().item() for n, p in model.named_parameters()
           if ".visual.blocks." in n and "lora_B" in n and p.grad is not None]
mrg_lora_g = [p.grad.abs().max().item() for n, p in model.named_parameters()
              if "merger" in n and "lora_B" in n and p.grad is not None]
mrg_full_g = [p.grad.abs().max().item() for n, p in model.named_parameters()
              if "merger" in n and ".modules_to_save." in n and n.endswith(".weight")
              and p.grad is not None]
lang_g = [p.grad.abs().max().item() for n, p in model.named_parameters()
          if ".visual." not in n and "lora_B" in n and p.grad is not None]
model.zero_grad(set_to_none=True)
assert lang_g and max(lang_g) > 0, "language LoRA got no gradients"
if TRAIN_VISION_LORA:
    assert len(tower_g) == 108 and min(tower_g) > 0, (
        f"tower LoRA dead or partial: {len(tower_g)}/108 tensors with grads")
if MERGER_MODE == "lora":
    assert len(mrg_lora_g) == 8 and min(mrg_lora_g) > 0,         f"merger LoRA dead or partial: {len(mrg_lora_g)}/8"
elif MERGER_MODE == "full":
    assert len(mrg_full_g) == 8 and min(mrg_full_g) > 0,         f"merger full weights dead or partial: {len(mrg_full_g)}/8"
else:
    assert not mrg_lora_g and not mrg_full_g, "merger got grads but knob=frozen"
print(f"grad gate OK (MERGER_MODE={MERGER_MODE}): tower min|g|={min(tower_g):.2e} "
      f"language max|g|={max(lang_g):.2e}")


In [ ]:
# Cell 6a — OPTIONAL throughput benchmark (flag-gated; run once, then set False)
# The A100 ran v18 at batch 1x8 with 13.8/40GB VRAM — headroom. This times
# fwd+bwd for candidate batch geometries at the REAL resolution so the long
# run uses the fastest safe config. To compare gradient-checkpointing modes
# ("unsloth" vs True), change it in cell 3 and rerun cells 3-6a once each.
RUN_THROUGHPUT_BENCH = False
BATCH_GEOMETRIES = [(1, 8), (2, 4), (4, 2)]   # (per_device_batch, grad_accum)

if RUN_THROUGHPUT_BENCH:
    import time
    FastVisionModel.for_training(model)
    bench_rows = [train_ds[i] for i in range(16)]   # map-style mixture: first 16 rows (all sources)
    for bs, ga in BATCH_GEOMETRIES:
        try:
            torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
            t0 = time.time(); n_micro = 0
            for step in range(2):                    # 2 optimizer steps
                for micro in range(ga):
                    rows = [bench_rows[(n_micro + i) % len(bench_rows)] for i in range(bs)]
                    b = {k: (v.to(model.device) if torch.is_tensor(v) else v)
                         for k, v in collator(rows).items()}
                    (model(**b).loss / ga).backward()
                    n_micro += 1
                model.zero_grad(set_to_none=True)
            dt = (time.time() - t0) / 2
            peak = torch.cuda.max_memory_allocated() / 1e9
            print(f"batch {bs}x{ga}: {dt:.1f}s/optimizer-step, peak {peak:.1f}GB")
        except torch.cuda.OutOfMemoryError:
            print(f"batch {bs}x{ga}: OOM — skip")
            torch.cuda.empty_cache()
    model.zero_grad(set_to_none=True)
    print("pick the fastest non-OOM geometry and set it in cell 6, then set "
          "RUN_THROUGHPUT_BENCH = False")

In [ ]:

# ---- v22b session helpers (inlined from src/finetuning/qwen_hebrew/train_session.py; keep in sync) ----
import gc, glob, hashlib, json, math, os, time
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Sequence, Tuple
import numpy as np
import torch
from torch.utils.data import Dataset as _TorchDataset, SequentialSampler
from transformers import TrainerCallback


REQUIRED_CHECKPOINT_FILES: Tuple[str, ...] = ("trainer_state.json", "optimizer.pt", "scheduler.pt")
WEIGHT_FILES: Tuple[str, ...] = ("adapter_model.safetensors", "adapter_model.bin", "model.safetensors", "pytorch_model.bin")
PLAN_IGNORED_KEYS: Tuple[str, ...] = ("order", "order_sha256", "created_at")


def materialize_order(n_rows: int, epochs: int, seed: int) -> List[int]:
    """Deterministic multi-epoch row order: one independent permutation per epoch, concatenated.

    :param n_rows: Dataset size.
    :type n_rows: int
    :param epochs: Number of passes to lay out (use the ceiling of ``max_steps × batch / n_rows``).
    :type epochs: int
    :param seed: Base seed; epoch ``e`` uses ``seed + e``.
    :type seed: int
    :return: ``n_rows × epochs`` dataset indices.
    :rtype: List[int]
    :raises ValueError: On non-positive sizes.
    """
    if n_rows <= 0 or epochs <= 0:
        raise ValueError("n_rows and epochs must be positive")
    order: List[int] = []
    for e in range(epochs):
        order.extend(np.random.default_rng(seed + e).permutation(n_rows).tolist())
    return order


def rows_consumed(global_step: int, per_device_batch: int, grad_accum: int, world_size: int = 1) -> int:
    """Rows the trainer has consumed after ``global_step`` optimizer steps.

    :param global_step: Restored ``trainer_state.global_step``.
    :type global_step: int
    :param per_device_batch: ``per_device_train_batch_size``.
    :type per_device_batch: int
    :param grad_accum: ``gradient_accumulation_steps``.
    :type grad_accum: int
    :param world_size: Number of processes (1 on Colab).
    :type world_size: int
    :return: Number of dataset rows already seen.
    :rtype: int
    """
    return global_step * per_device_batch * grad_accum * world_size


def remaining_rows(order: Sequence[int], global_step: int, per_device_batch: int, grad_accum: int,
                   world_size: int = 1) -> List[int]:
    """The tail of the materialised order a resumed session still has to train on.

    :param order: Output of :func:`materialize_order`.
    :type order: Sequence[int]
    :param global_step: Restored ``global_step`` (0 for a fresh run).
    :type global_step: int
    :param per_device_batch: ``per_device_train_batch_size``.
    :type per_device_batch: int
    :param grad_accum: ``gradient_accumulation_steps``.
    :type grad_accum: int
    :param world_size: Number of processes.
    :type world_size: int
    :return: Remaining indices (possibly empty when the order is exhausted).
    :rtype: List[int]
    """
    done = rows_consumed(global_step, per_device_batch, grad_accum, world_size)
    return list(order[done:])


def epochs_for(max_steps: int, n_rows: int, per_device_batch: int, grad_accum: int, world_size: int = 1) -> int:
    """Smallest number of passes whose row count covers ``max_steps`` optimizer steps.

    :param max_steps: Planned optimizer steps.
    :type max_steps: int
    :param n_rows: Dataset size.
    :type n_rows: int
    :param per_device_batch: ``per_device_train_batch_size``.
    :type per_device_batch: int
    :param grad_accum: ``gradient_accumulation_steps``.
    :type grad_accum: int
    :param world_size: Number of processes.
    :type world_size: int
    :return: Passes (at least 1).
    :rtype: int
    """
    need = rows_consumed(max_steps, per_device_batch, grad_accum, world_size)
    return max(1, -(-need // n_rows))


@dataclass
class BudgetState:
    """What the time-budget callback decided, for logging and tests.

    :param stopped: Whether training was ended by the budget.
    :param elapsed_s: Elapsed wall-clock seconds when the decision was taken.
    :param step: ``global_step`` at that point.
    """
    stopped: bool = False
    elapsed_s: float = 0.0
    step: int = 0


class TimeBudgetCallback:
    """Stop training cleanly (save + push) when a wall-clock budget is spent.

    Implemented against the ``transformers.TrainerCallback`` interface without importing
    transformers at module import time, so the pure logic is unit-testable on a CPU box.
    Attach with ``trainer.add_callback(TimeBudgetCallback(budget_s=21.5 * 3600))``.

    :param budget_s: Wall-clock budget in seconds measured from the first :meth:`on_train_begin`
        (from callback creation until then).
    :type budget_s: float
    :param min_steps_between_saves: Do not stop within this many steps of the last regular
        save unless the budget is exhausted anyway (avoids a redundant checkpoint).
    :type min_steps_between_saves: int
    :param clock: Time source (injectable for tests).
    """

    def __init__(self, budget_s: float, min_steps_between_saves: int = 0, clock=time.monotonic) -> None:
        self.budget_s = float(budget_s)
        self.min_steps_between_saves = int(min_steps_between_saves)
        self._clock = clock
        self._t0 = clock()
        self._started = False
        self.state = BudgetState()

    def on_train_begin(self, args=None, state=None, control=None, **kwargs):
        """Start the clock when training first starts.

        Only the first call resets it: an in-session retry calls ``train()`` again, and the
        budget has to keep counting from the session's first step or the retry would run past
        Colab's ceiling.

        :return: ``control`` unchanged.
        """
        if not self._started:
            self._t0 = self._clock()
            self._started = True
        return control

    def elapsed(self) -> float:
        """Seconds since the clock was (re)started.

        :return: Elapsed seconds.
        :rtype: float
        """
        return self._clock() - self._t0

    def should_stop(self, global_step: int) -> bool:
        """Pure decision: budget spent?

        :param global_step: Current optimizer step.
        :type global_step: int
        :return: True when training should end now.
        :rtype: bool
        """
        return self.elapsed() >= self.budget_s

    def on_step_end(self, args=None, state=None, control=None, **kwargs):
        """Request save + stop at the end of the step that crosses the budget.

        :return: ``control`` with ``should_save`` and ``should_training_stop`` set when stopping.
        """
        step = int(getattr(state, "global_step", 0) or 0)
        if not self.state.stopped and self.should_stop(step):
            self.state = BudgetState(stopped=True, elapsed_s=self.elapsed(), step=step)
            if control is not None:
                control.should_save = True
                control.should_training_stop = True
            print(f"[TimeBudgetCallback] budget {self.budget_s/3600:.1f} h spent at step {step}; saving and stopping", flush=True)
        return control


def describe_session(n_rows: int, max_steps: int, per_device_batch: int, grad_accum: int, global_step: int,
                     seed: int, world_size: int = 1) -> str:
    """Human-readable summary of what a session will train on.

    :param n_rows: Dataset size.
    :type n_rows: int
    :param max_steps: Planned optimizer steps.
    :type max_steps: int
    :param per_device_batch: ``per_device_train_batch_size``.
    :type per_device_batch: int
    :param grad_accum: ``gradient_accumulation_steps``.
    :type grad_accum: int
    :param global_step: Restored step (0 = fresh).
    :type global_step: int
    :param seed: Order seed.
    :type seed: int
    :param world_size: Processes.
    :type world_size: int
    :return: One-line description.
    :rtype: str
    """
    ep = epochs_for(max_steps, n_rows, per_device_batch, grad_accum, world_size)
    order = materialize_order(n_rows, ep, seed)
    rem = remaining_rows(order, global_step, per_device_batch, grad_accum, world_size)
    steps_left = len(rem) // (per_device_batch * grad_accum * world_size)
    return (f"{n_rows} rows × {ep} pass(es) = {len(order)} ordered rows; resumed at step {global_step} → "
            f"{len(rem)} rows / {steps_left} steps remain (target max_steps {max_steps}); seed {seed}")


def warmup_steps_for(max_steps: int, warmup_ratio: float) -> int:
    """Warm-up steps the trainer derives from a ratio (``TrainingArguments.get_warmup_steps``).

    :param max_steps: Planned optimizer steps.
    :type max_steps: int
    :param warmup_ratio: ``warmup_ratio`` training argument.
    :type warmup_ratio: float
    :return: ``ceil(max_steps × warmup_ratio)``.
    :rtype: int
    """
    return math.ceil(max_steps * warmup_ratio)


def cosine_warmup_lr(step: int, base_lr: float, warmup_steps: int, total_steps: int) -> float:
    """Learning rate after ``step`` scheduler steps under linear warm-up + half-cosine decay.

    Mirrors ``transformers.get_cosine_schedule_with_warmup`` (``num_cycles=0.5``): this is the
    rate the optimizer uses for optimizer step ``step + 1``, and what a correctly restored
    scheduler reports right after ``resume_from_checkpoint`` at ``global_step == step``.

    :param step: Scheduler steps taken so far (the restored ``global_step``).
    :type step: int
    :param base_lr: ``learning_rate`` training argument.
    :type base_lr: float
    :param warmup_steps: Warm-up length in steps.
    :type warmup_steps: int
    :param total_steps: ``max_steps``.
    :type total_steps: int
    :return: Scheduled learning rate.
    :rtype: float
    """
    if step < warmup_steps:
        return base_lr * step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return base_lr * max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))


def order_fingerprint(order: Sequence[int]) -> str:
    """SHA-256 of a row order (detects a changed permutation between sessions).

    :param order: Dataset indices in training order.
    :type order: Sequence[int]
    :return: Hex digest.
    :rtype: str
    """
    return hashlib.sha256(",".join(str(int(i)) for i in order).encode("ascii")).hexdigest()


def rows_fingerprint(columns: Sequence[Sequence[object]]) -> str:
    """SHA-256 over the rows of a dataset, given some of its columns (index → row mapping check).

    :param columns: Equal-length column lists (for example image hash, source, task, question, answer).
    :type columns: Sequence[Sequence[object]]
    :return: Hex digest.
    :rtype: str
    :raises ValueError: When the columns differ in length.
    """
    lengths = {len(c) for c in columns}
    if len(lengths) > 1:
        raise ValueError(f"columns differ in length: {sorted(lengths)}")
    digest = hashlib.sha256()
    for row in zip(*columns):
        digest.update(("\x1f".join("" if v is None else str(v) for v in row) + "\x1e").encode("utf-8"))
    return digest.hexdigest()


def build_session_plan(order: Sequence[int], **schedule: object) -> Dict[str, object]:
    """Everything that defines the training schedule, as one JSON-serialisable record.

    :param order: The materialised row order the run trains on.
    :type order: Sequence[int]
    :param schedule: Schedule-defining settings (steps, batch, accumulation, learning rate,
        warm-up, scheduler, optimiser, seeds, dataset repo and revision, row fingerprint, ...).
    :type schedule: object
    :return: The settings plus ``order_len`` and ``order_sha256``.
    :rtype: Dict[str, object]
    """
    plan: Dict[str, object] = dict(schedule)
    plan["order_len"] = len(order)
    plan["order_sha256"] = order_fingerprint(order)
    return plan


def diff_session_plan(stored: Dict[str, object], current: Dict[str, object],
                      ignore: Iterable[str] = PLAN_IGNORED_KEYS) -> List[str]:
    """Differences between the plan stored at the first launch and this session's plan.

    :param stored: Plan read back from the checkpoint repository.
    :type stored: Dict[str, object]
    :param current: Plan built from this session's settings.
    :type current: Dict[str, object]
    :param ignore: Keys that do not define the schedule (the order itself is taken from the stored
        plan, so its fingerprint is not compared here).
    :type ignore: Iterable[str]
    :return: One line per differing key; empty when the schedules are the same.
    :rtype: List[str]
    """
    skip = set(ignore)
    missing = object()
    problems: List[str] = []
    for key in sorted((set(stored) | set(current)) - skip):
        a, b = stored.get(key, missing), current.get(key, missing)
        if a is missing:
            problems.append(f"{key}: not in the stored plan, now {b!r}")
        elif b is missing:
            problems.append(f"{key}: stored {a!r}, missing now")
        elif a != b:
            problems.append(f"{key}: stored {a!r}, now {b!r}")
    return problems


def checkpoint_step(path: str) -> Optional[int]:
    """``global_step`` of a checkpoint folder that can restore the full training state.

    :param path: Checkpoint directory.
    :type path: str
    :return: The step, or ``None`` when trainer state, optimizer, scheduler or weights are missing
        or the trainer state does not parse (a folder still being written or a partial download).
    :rtype: Optional[int]
    """
    if not all(os.path.isfile(os.path.join(path, f)) for f in REQUIRED_CHECKPOINT_FILES):
        return None
    if not any(os.path.isfile(os.path.join(path, f)) for f in WEIGHT_FILES):
        return None
    with open(os.path.join(path, "trainer_state.json"), encoding="utf-8") as fh:
        text = fh.read()
    if not text.strip().endswith("}"):
        return None
    step = json.loads(text).get("global_step")
    return int(step) if isinstance(step, int) else None


def local_checkpoints(output_dir: str) -> List[str]:
    """The trainer's ``checkpoint-*`` folders under an output directory.

    :param output_dir: ``TrainingArguments.output_dir``.
    :type output_dir: str
    :return: Sorted directory paths (possibly empty).
    :rtype: List[str]
    """
    return sorted(d for d in glob.glob(os.path.join(output_dir, "checkpoint-*")) if os.path.isdir(d))


def newest_complete_checkpoint(paths: Sequence[str]) -> Tuple[Optional[str], int]:
    """The complete checkpoint with the highest ``global_step``.

    :param paths: Candidate checkpoint directories (missing or incomplete ones are skipped).
    :type paths: Sequence[str]
    :return: ``(path, step)``; ``(None, 0)`` when no candidate is complete.
    :rtype: Tuple[Optional[str], int]
    """
    best: Tuple[Optional[str], int] = (None, 0)
    for path in paths:
        if not os.path.isdir(path):
            continue
        step = checkpoint_step(path)
        if step is not None and (best[0] is None or step > best[1]):
            best = (path, step)
    return best


def resume_problems(global_step: int, expected_step: int, scheduler_step: Optional[int], lr: float,
                    expected_lr: float, n_batches: int, expected_batches: int, state_max_steps: int,
                    max_steps: int, optimizer_steps: Optional[Sequence[int]], rel_tol: float = 1e-6) -> List[str]:
    """Check the state a trainer is about to train from against the planned schedule.

    :param global_step: ``trainer.state.global_step`` after the checkpoint was loaded.
    :type global_step: int
    :param expected_step: Step of the checkpoint the session resumes from (0 = fresh).
    :type expected_step: int
    :param scheduler_step: ``lr_scheduler.last_epoch`` (``None`` when not exposed).
    :type scheduler_step: Optional[int]
    :param lr: Learning rate the scheduler reports now.
    :type lr: float
    :param expected_lr: :func:`cosine_warmup_lr` at ``expected_step``.
    :type expected_lr: float
    :param n_batches: Length of the prepared train dataloader.
    :type n_batches: int
    :param expected_batches: Batches left in the fixed row order.
    :type expected_batches: int
    :param state_max_steps: ``trainer.state.max_steps``.
    :type state_max_steps: int
    :param max_steps: Planned ``max_steps``.
    :type max_steps: int
    :param optimizer_steps: Per-parameter step counters of the optimizer (``None`` when the
        optimizer does not expose them, which is not treated as a problem).
    :type optimizer_steps: Optional[Sequence[int]]
    :param rel_tol: Relative tolerance on the learning rate.
    :type rel_tol: float
    :return: One line per mismatch; empty when the state is exactly the planned one.
    :rtype: List[str]
    """
    problems: List[str] = []
    if global_step != expected_step:
        problems.append(f"global_step is {global_step}, the checkpoint is step {expected_step}")
    if scheduler_step is not None and scheduler_step != expected_step:
        problems.append(f"the LR scheduler is at step {scheduler_step}, expected {expected_step} (schedule not restored)")
    if abs(lr - expected_lr) > rel_tol * max(abs(expected_lr), 1e-12):
        problems.append(f"learning rate is {lr:.6e}, the schedule gives {expected_lr:.6e} at step {expected_step}")
    if n_batches != expected_batches:
        problems.append(f"the train dataloader has {n_batches} batches, the fixed order has {expected_batches} left")
    if state_max_steps != max_steps:
        problems.append(f"the trainer will stop at step {state_max_steps}, the plan says {max_steps}")
    if optimizer_steps is not None:
        top = max(optimizer_steps) if optimizer_steps else 0
        if expected_step > 0 and not optimizer_steps:
            problems.append("the optimizer has no state: its moments were not restored from the checkpoint")
        elif top != expected_step:
            problems.append(f"the optimizer has taken {top} steps, expected {expected_step} (optimizer state not restored)")
    return problems


def may_retry(attempt: int, max_retries: int, failed_step: int, last_failed_step: Optional[int],
              checkpoint: Optional[int]) -> bool:
    """Whether a failed ``train()`` call may be re-entered inside the same session.

    :param attempt: Retries already used in this session.
    :type attempt: int
    :param max_retries: Retry budget.
    :type max_retries: int
    :param failed_step: ``global_step`` when the failure happened.
    :type failed_step: int
    :param last_failed_step: ``global_step`` of the previous failure in this session, if any.
    :type last_failed_step: Optional[int]
    :param checkpoint: Step of the newest complete checkpoint, or ``None`` when there is none.
    :type checkpoint: Optional[int]
    :return: True when retries remain, a complete checkpoint exists to restart from (without one
        the weights in memory are no longer the planned ones), and the failure is not a repeat at
        the same step (a row or state that fails deterministically needs a person).
    :rtype: bool
    """
    return attempt < max_retries and checkpoint is not None and failed_step != last_failed_step


# ---- notebook-only glue (needs torch / transformers / the Hub) ----
PLAN_FILE = "session_plan.json"


class ResumeGateError(AssertionError):
    """The state about to be trained from is not the planned schedule. Never retried."""


class OrderedView(_TorchDataset):
    """A map-style view of ``base`` in a fixed index order (the rows a session still has to consume)."""

    def __init__(self, base, indices):
        self.base, self.indices = base, list(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        return self.base[self.indices[i]]

    def column(self, name):
        col = self.base.column(name)
        return [col[j] for j in self.indices]


class _RecordingView(_TorchDataset):
    """Records the positions a dataloader fetches (order gate)."""

    def __init__(self, ds):
        self.ds, self.seen = ds, []

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, i):
        self.seen.append(int(i))
        return self.ds[i]


class TimeBudgetTrainerCallback(TrainerCallback):
    """``TimeBudgetCallback`` as a real ``TrainerCallback`` (the core stays framework-free for tests)."""

    def __init__(self, budget_s: float):
        super().__init__()
        self.core = TimeBudgetCallback(budget_s)

    def on_train_begin(self, args, state, control, **kwargs):
        return self.core.on_train_begin(args, state, control, **kwargs)

    def on_step_end(self, args, state, control, **kwargs):
        return self.core.on_step_end(args, state, control, **kwargs)


def _optimizer_steps(optimizer) -> Optional[List[int]]:
    """Per-parameter step counters of an optimizer (AdamW and the 8-bit AdamW both keep ``state[p]["step"]``).

    :param optimizer: The trainer's optimizer (possibly wrapped by Accelerate).
    :return: Step counters, or ``None`` when the optimizer does not expose them.
    """
    state = getattr(getattr(optimizer, "optimizer", optimizer), "state", None)
    if state is None:
        return None
    steps = []
    for entry in state.values():
        if isinstance(entry, dict) and "step" in entry:
            value = entry["step"]
            steps.append(int(value.item()) if hasattr(value, "item") else int(value))
    return steps


class ResumeGateTrainerCallback(TrainerCallback):
    """Before the first step of every ``train()`` call: the restored state must be the planned schedule.

    Runs in ``on_train_begin``, i.e. after the trainer loaded weights, optimizer and scheduler from the
    checkpoint and built its train dataloader. Raises :class:`ResumeGateError` on any mismatch.
    """

    def __init__(self):
        super().__init__()
        self.expected_step, self.expected_rows, self.passed = 0, 0, []

    def expect(self, step: int, rows: int) -> None:
        """Set what the next ``train()`` call must start from.

        :param step: Step of the checkpoint (0 = fresh).
        :param rows: Rows left in the fixed order.
        """
        self.expected_step, self.expected_rows = int(step), int(rows)

    def on_train_begin(self, args, state, control, optimizer=None, lr_scheduler=None, train_dataloader=None, **kwargs):
        kind = str(getattr(args.lr_scheduler_type, "value", args.lr_scheduler_type))
        if kind != "cosine":
            raise ResumeGateError(f"the resume gate knows the cosine schedule, the trainer uses {kind!r}")
        inner = getattr(optimizer, "optimizer", optimizer)
        lr = float(inner.param_groups[0]["lr"])
        expected_lr = cosine_warmup_lr(self.expected_step, args.learning_rate, args.get_warmup_steps(args.max_steps), args.max_steps)
        counters = _optimizer_steps(optimizer)
        problems = resume_problems(
            global_step=int(state.global_step), expected_step=self.expected_step,
            scheduler_step=getattr(lr_scheduler, "last_epoch", None), lr=lr, expected_lr=expected_lr,
            n_batches=len(train_dataloader), expected_batches=-(-self.expected_rows // args.per_device_train_batch_size),
            state_max_steps=int(state.max_steps), max_steps=int(args.max_steps), optimizer_steps=counters)
        if problems:
            raise ResumeGateError("resume gate FAILED before the first step:\n  - " + "\n  - ".join(problems))
        self.passed.append(self.expected_step)
        seen = "not exposed" if counters is None else (f"{len(counters)} tensors at step {max(counters)}" if counters else "empty (fresh)")
        print(f"resume gate OK: step {state.global_step} of {args.max_steps}, lr {lr:.6e} (schedule {expected_lr:.6e}), "
              f"scheduler at {getattr(lr_scheduler, 'last_epoch', '?')}, optimizer state {seen}, {self.expected_rows} rows left", flush=True)
        return control


def order_gate(trainer, view, k: int = 3) -> None:
    """The PREPARED train dataloader must hand out the view's first rows, in order.

    :param trainer: The trainer (its sampler override is already installed).
    :param view: The ordered view it is about to train on.
    :param k: Rows to check.
    :raises ResumeGateError: When the dataloader fetches anything but positions ``0..k-1``.
    """
    rec = _RecordingView(view)
    saved_ds, saved_workers = trainer.train_dataset, trainer.args.dataloader_num_workers
    trainer.train_dataset, trainer.args.dataloader_num_workers = rec, 0   # in-process fetch: positions visible, nothing prefetched
    it = iter(trainer.get_train_dataloader())
    n = min(k, len(view))
    while len(rec.seen) < n:
        next(it)
    trainer.train_dataset, trainer.args.dataloader_num_workers = saved_ds, saved_workers
    del it
    if rec.seen[:n] != list(range(n)):
        raise ResumeGateError(f"order gate FAILED: the dataloader fetched view positions {rec.seen[:n]}, expected {list(range(n))} "
                              "(the sequential sampler is not in effect)")
    print(f"order gate OK: the prepared dataloader starts with rows {[view.indices[i] for i in range(n)]} of the fixed order")


def train_with_exact_resume(trainer, base_ds, order, resume_dir, gate, max_retries: int = 2) -> int:
    """Train over the fixed row order from ``resume_dir`` (or step 0); re-enter after an in-session failure.

    Every attempt starts from a complete checkpoint: weights, optimizer and scheduler are reloaded by the
    trainer, the dataset is the tail of ``order`` that checkpoint has not consumed, the sampler is
    sequential, and both gates run before the first step. A failure is retried at most ``max_retries``
    times, never without a checkpoint and never twice at the same step.

    :param trainer: The trainer (``max_steps`` set; callbacks ``gate`` and the time budget already added).
    :param base_ds: The full train dataset (map-style).
    :param order: The materialised row order of the whole run.
    :param resume_dir: Checkpoint to start from, or ``None`` for a fresh run.
    :param gate: The :class:`ResumeGateTrainerCallback` attached to ``trainer``.
    :param max_retries: In-session retry budget.
    :return: ``global_step`` when training returned.
    """
    batch, accum = trainer.args.per_device_train_batch_size, trainer.args.gradient_accumulation_steps
    attempt, last_failed_step = 0, None
    while True:
        step = 0
        if resume_dir is not None:
            step = checkpoint_step(resume_dir)
            if step is None:
                raise ResumeGateError(f"{resume_dir} is not a complete checkpoint (trainer state, optimizer, scheduler and weights are all needed)")
        remaining = remaining_rows(order, step, batch, accum)
        if step >= trainer.args.max_steps or not remaining:
            trainer._load_from_checkpoint(resume_dir)
            print(f"run already complete at step {step}: weights loaded from {resume_dir}, nothing left to train")
            return step
        view = OrderedView(base_ds, remaining)
        trainer.train_dataset = view
        trainer.args.ignore_data_skip = True     # the skip is done by construction: the view holds only unseen rows
        trainer._get_train_sampler = lambda *a, **k: SequentialSampler(trainer.train_dataset)   # the order is baked into the view
        order_gate(trainer, view)
        gate.expect(step, len(view))
        print(f"train() attempt {attempt + 1}: from step {step}, {len(view)} rows left in the fixed order", flush=True)
        failure = None
        try:
            trainer.train(resume_from_checkpoint=resume_dir)
        except Exception as error:   # the one deliberate catch-all: a long run re-enters from its last checkpoint
            if isinstance(error, AssertionError):
                raise                # gate failures are never retried
            failure = error
        if failure is None:
            return int(trainer.state.global_step)
        failed_step = int(trainer.state.global_step)
        candidates = local_checkpoints(trainer.args.output_dir) + ([resume_dir] if resume_dir else [])
        next_dir, next_step = newest_complete_checkpoint(candidates)
        if not may_retry(attempt, max_retries, failed_step, last_failed_step, next_step if next_dir else None):
            raise failure
        print(f"⚠️ train() failed at step {failed_step}: {type(failure).__name__}: {str(failure)[:300]}\n"
              f"   in-session retry {attempt + 1}/{max_retries} from {next_dir} (step {next_step})", flush=True)
        attempt, last_failed_step, resume_dir, failure = attempt + 1, failed_step, next_dir, None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def discover_resume(ckpt_repo: str, out_dir: str, resume_root: str):
    """Newest complete checkpoint among the Hub's ``last-checkpoint`` and this machine's ``checkpoint-*`` folders.

    The Hub copy is downloaded to ``resume_root``, which must lie OUTSIDE ``out_dir``: the trainer's model
    push uploads everything in ``output_dir`` except ``checkpoint-*``, so a ``last-checkpoint`` folder
    inside it is re-uploaded at every save and rolls the Hub's resume point back to the session's start.
    Only "the repository does not exist" means a fresh start; any other Hub error propagates.

    :param ckpt_repo: Checkpoint repository id.
    :param out_dir: The trainer's ``output_dir``.
    :param resume_root: Download directory for the Hub checkpoint.
    :return: ``(checkpoint_dir or None, step)``.
    """
    from huggingface_hub import list_repo_files, snapshot_download
    from huggingface_hub.utils import RepositoryNotFoundError
    out_abs, root_abs = os.path.abspath(out_dir), os.path.abspath(resume_root)
    if root_abs == out_abs or root_abs.startswith(out_abs + os.sep):
        raise ResumeGateError(f"resume_root {resume_root} must not be inside output_dir {out_dir}")
    candidates = local_checkpoints(out_dir)
    try:
        files = list_repo_files(ckpt_repo)
    except RepositoryNotFoundError:
        files = []
        print(f"{ckpt_repo} does not exist yet")
    on_hub = any(f.startswith("last-checkpoint/") for f in files)
    if on_hub:
        snapshot_download(ckpt_repo, allow_patterns="last-checkpoint/*", local_dir=resume_root)
        candidates.append(os.path.join(resume_root, "last-checkpoint"))
    path, step = newest_complete_checkpoint(candidates)
    if on_hub and path is None:
        raise ResumeGateError(f"{ckpt_repo} has a last-checkpoint but no complete copy could be assembled (optimizer / scheduler / weights missing)")
    return path, step


def lock_session_plan(ckpt_repo: str, plan: Dict[str, object], order: Sequence[int], has_checkpoint: bool) -> List[int]:
    """Store the schedule and row order on the first launch; afterwards require the same schedule.

    :param ckpt_repo: Checkpoint repository id (exists: the trainer creates it).
    :param plan: This session's :func:`build_session_plan`.
    :param order: The row order materialised in this session.
    :param has_checkpoint: Whether a checkpoint was found to resume from.
    :return: The order to train on (the stored one from the second session on).
    :raises ResumeGateError: On any schedule difference, or a checkpoint without a stored plan.
    """
    from huggingface_hub import HfApi, hf_hub_download
    from huggingface_hub.utils import EntryNotFoundError
    try:
        with open(hf_hub_download(ckpt_repo, PLAN_FILE, force_download=True), encoding="utf-8") as fh:
            stored = json.load(fh)
    except EntryNotFoundError:
        stored = None
    if stored is None:
        if has_checkpoint:
            raise ResumeGateError(f"{ckpt_repo} has a checkpoint but no {PLAN_FILE}: it was not started by this notebook, refusing to continue it")
        payload = dict(plan, order=[int(i) for i in order], created_at=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
        HfApi().upload_file(path_or_fileobj=json.dumps(payload).encode("utf-8"), path_in_repo=PLAN_FILE, repo_id=ckpt_repo,
                            commit_message="Session plan: schedule and fixed row order (exact resume)")
        print(f"session plan stored in {ckpt_repo}/{PLAN_FILE} ({len(order)} ordered rows, sha {plan['order_sha256'][:12]})")
        return [int(i) for i in order]
    problems = diff_session_plan(stored, plan)
    if problems:
        raise ResumeGateError("this session's settings differ from the plan the run was started with:\n  - " + "\n  - ".join(problems))
    stored_order = [int(i) for i in stored["order"]]
    if order_fingerprint(stored_order) != stored["order_sha256"] or len(stored_order) != stored["order_len"]:
        raise ResumeGateError(f"the row order stored in {PLAN_FILE} does not match its own fingerprint")
    if stored["order_sha256"] != plan["order_sha256"]:
        print("⚠️ this session regenerated a different permutation (library change); training on the STORED order")
    print(f"session plan verified against {ckpt_repo}/{PLAN_FILE} (started {stored.get('created_at', '?')})")
    return stored_order


def hub_checkpoint_step(ckpt_repo: str) -> Optional[int]:
    """``global_step`` of the Hub's ``last-checkpoint`` right now.

    :param ckpt_repo: Checkpoint repository id.
    :return: The step, or ``None`` when there is no ``last-checkpoint``.
    """
    from huggingface_hub import hf_hub_download
    from huggingface_hub.utils import EntryNotFoundError
    try:
        path = hf_hub_download(ckpt_repo, "last-checkpoint/trainer_state.json", force_download=True)
    except EntryNotFoundError:
        return None
    with open(path, encoding="utf-8") as fh:
        return int(json.load(fh)["global_step"])


def ensure_hub_checkpoint(ckpt_repo: str, out_dir: str, expected_step: int) -> None:
    """The session may only end once the Hub's resume point is the step training stopped on.

    :param ckpt_repo: Checkpoint repository id.
    :param out_dir: The trainer's ``output_dir``.
    :param expected_step: ``global_step`` when ``train()`` returned.
    :raises ResumeGateError: When the checkpoint cannot be confirmed on the Hub.
    """
    step = hub_checkpoint_step(ckpt_repo)
    if step != expected_step:
        from huggingface_hub import upload_folder
        path, local_step = newest_complete_checkpoint(local_checkpoints(out_dir))
        if path is None or local_step != expected_step:
            raise ResumeGateError(f"the Hub's last-checkpoint is step {step}, training stopped at {expected_step}, "
                                  f"and no complete local checkpoint of that step exists (newest: {path} at {local_step})")
        print(f"⚠️ the Hub's last-checkpoint is step {step}, training stopped at {expected_step}: pushing {path} now", flush=True)
        upload_folder(repo_id=ckpt_repo, folder_path=path, path_in_repo="last-checkpoint",
                      commit_message=f"Training in progress, step {expected_step}, checkpoint (verified re-push)")
        step = hub_checkpoint_step(ckpt_repo)
    if step != expected_step:
        raise ResumeGateError(f"the Hub's last-checkpoint is step {step} after the push, expected {expected_step}")
    print(f"Hub resume point verified: last-checkpoint is step {step}")


MAX_STEPS = 1500
TIME_BUDGET_H = 21.5
MAX_INSESSION_RETRIES = 2
ORDER_SEED = 3407
print(f"session helpers ready: MAX_STEPS={MAX_STEPS}, time budget {TIME_BUDGET_H} h, in-session retries {MAX_INSESSION_RETRIES}")


In [ ]:
# Cell 6 — v2.3a (Arabic script) training: materialised map-style mixture + per-source eval
import inspect
import wandb
from huggingface_hub import list_repo_files
from torch.utils.data import Dataset as _TDataset
from trl import SFTConfig, SFTTrainer
from unsloth import is_bf16_supported

CKPT_REPO = "isaacmg/qwen3-vl-8b-hebrew-v23a-ckpt"  # NEW repo: never reuse another run's (auto-resume would load its last-checkpoint)
OUT_DIR = "outputs_v23a"
wandb.login(key=userdata.get("WANDB_API_KEY"))

def make_sft_config(**kw):
    params = inspect.signature(SFTConfig.__init__).parameters
    if "max_seq_length" in kw and "max_seq_length" not in params:
        kw["max_length"] = kw.pop("max_seq_length")
    dropped = {k: kw.pop(k) for k in list(kw) if k not in params}
    if dropped:
        print(f"⚠️ dropped unsupported SFTConfig kwargs: {sorted(dropped)}")
    return SFTConfig(**kw)

def make_trainer(**kw):
    try:
        return SFTTrainer(**kw)
    except TypeError as e:
        if "tokenizer" in kw and ("tokenizer" in str(e) or "processing_class" in str(e)):
            kw["processing_class"] = kw.pop("tokenizer")
            return SFTTrainer(**kw)
        raise

# ---- v22b: where to resume from (helpers cell) ----
# The Hub's last-checkpoint is downloaded OUTSIDE output_dir: inside it, every model push re-uploads the stale folder
# and rolls the Hub's resume point back to this session's start (seen on the v19c / v20a / v21b resumed sessions).
# Only "the repository does not exist" is a fresh start; any other Hub error stops the notebook here.
RESUME_ROOT = "/content/v23a_resume"
resume_dir, _resume_step = discover_resume(CKPT_REPO, OUT_DIR, RESUME_ROOT)
print(f"resume point: {resume_dir} (step {_resume_step})" if resume_dir else "no checkpoint on the Hub or on this machine: fresh start from the warm start")

FastVisionModel.for_training(model)
trainer = make_trainer(
    model=model, tokenizer=tokenizer, data_collator=collator,
    train_dataset=train_ds, eval_dataset=eval_ds,
    args=make_sft_config(
        per_device_train_batch_size=1, gradient_accumulation_steps=8,   # 1x8 proven on the A100 (~16/40GB, ~65 s/step)
        dataloader_num_workers=2, dataloader_pin_memory=True,
        max_steps=MAX_STEPS,               # 12,000 rows = one pass = 1,500 steps; sessions end on the time budget and resume exactly
        learning_rate=1e-4,                # a new script has to be learned; v22b's 3e-5 only adjusted a converged model
        warmup_ratio=0.05, lr_scheduler_type="cosine", weight_decay=0.01,
        logging_steps=10,
        eval_strategy="steps", eval_steps=100, per_device_eval_batch_size=1,
        save_steps=100, save_total_limit=2,
        push_to_hub=True, hub_model_id=CKPT_REPO,
        hub_strategy="checkpoint", hub_private_repo=True,    # Muharaf is CC BY-NC-SA: private until the licence question is decided
        hub_always_push=True,              # every save is pushed, also the time-budget stop that lands right after a regular save
        optim="adamw_8bit", seed=3407, data_seed=3407, output_dir=OUT_DIR,
        report_to="wandb", run_name="genizah_v23a",
        bf16=is_bf16_supported(), fp16=not is_bf16_supported(),
        remove_unused_columns=False, dataset_text_field="",
        # map-style train set: Accelerate does not dispatch, but the v2.1 rule stays explicit and gated
        accelerator_config={"dispatch_batches": False},
        dataset_kwargs={"skip_prepare_dataset": True},
        max_seq_length=MAX_SEQ,
    ),
)
wandb.init(project="qwen-hebrew-finetune", name="genizah_v23a", config={"resume_step": _resume_step, "v22_repo": V22_REPO, "v22_revision": V22_REVISION, "warm_ckpt": f"{WARM_CKPT_REPO}@{WARM_REVISION}",
                     "mixture": MIX.get("config", {})}, allow_val_change=True)

# ---- DATALOADER GATE (v2.1 lesson, kept for map-style) — checks the PREPARED train dataloader, not the collator ----
assert trainer.args.accelerator_config.dispatch_batches is False, "dispatch_batches=False did not reach the trainer"

class _Recording(_TDataset):
    """Wrapper that records fetched indices so the gate can find the row behind a prepared batch."""
    def __init__(self, ds):
        self.ds, self.seen = ds, []
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, i):
        self.seen.append(int(i))
        return self.ds[i]

_rec = _Recording(train_ds)
trainer.train_dataset = _rec
_saved_workers, trainer.args.dataloader_num_workers = trainer.args.dataloader_num_workers, 0  # in-process fetch ⇒ indices visible
_dl = trainer.get_train_dataloader()
assert "Dispatcher" not in type(_dl).__name__, f"train dataloader is {type(_dl).__name__} — batches would be truncated"
_it = iter(_dl)
for _k in range(2):
    _b = next(_it)
    _need = int(_b["image_grid_thw"].prod(dim=-1).sum())
    assert _b["pixel_values"].shape[0] == _need, (
        f"batch {_k}: pixel_values has {_b['pixel_values'].shape[0]} patch rows, grid needs {_need} — truncated")
    if _k == 0:
        _first, _idx = _b, _rec.seen[0]
# the loss on a prepared batch must equal the loss on the same row collated directly (image intact end to end)
_row = train_ds[_idx]
_direct = {k: (v.to(model.device) if torch.is_tensor(v) else v) for k, v in collator([_row]).items()}
assert torch.equal(_direct["input_ids"], _first["input_ids"].to(model.device)), (
    f"prepared batch 0 is not train row {_idx} — the recording wrapper and the sampler disagree")
model.eval()
with torch.no_grad():
    _lp = model(**{k: (v.to(model.device) if torch.is_tensor(v) else v) for k, v in _first.items()}).loss.item()
    _ld = model(**_direct).loss.item()
assert abs(_lp - _ld) <= 0.05 * max(1.0, _ld), f"prepared-dataloader loss {_lp:.3f} != direct-collate loss {_ld:.3f}"
_eb = next(iter(trainer.get_eval_dataloader()))
assert _eb["pixel_values"].shape[0] == int(_eb["image_grid_thw"].prod(dim=-1).sum()), "eval batch truncated"
print(f"dataloader gate OK: {type(_dl).__name__}, patches intact, row {_idx} ({_row['source']}/{_row['task']}), "
      f"loss prepared={_lp:.3f} direct={_ld:.3f}")
trainer.train_dataset = train_ds
trainer.args.dataloader_num_workers = _saved_workers
del _dl, _it, _b, _first, _row, _direct, _eb, _rec
FastVisionModel.for_training(model)
# ---- v22b: schedule lock, fixed row order, exact resume (helpers cell) ----
_a = trainer.args
_order = materialize_order(len(train_ds), epochs_for(MAX_STEPS, len(train_ds), _a.per_device_train_batch_size, _a.gradient_accumulation_steps), seed=ORDER_SEED)
_plan = build_session_plan(
    _order, order_seed=ORDER_SEED, n_rows=len(train_ds), max_steps=_a.max_steps,
    per_device_batch=_a.per_device_train_batch_size, grad_accum=_a.gradient_accumulation_steps,
    learning_rate=_a.learning_rate, warmup_ratio=_a.warmup_ratio,
    lr_scheduler_type=str(getattr(_a.lr_scheduler_type, "value", _a.lr_scheduler_type)),
    weight_decay=_a.weight_decay, optim=str(getattr(_a.optim, "value", _a.optim)),
    dataset_repo=V22_REPO, dataset_revision=V22_REVISION, warm_start=f"{WARM_CKPT_REPO}@{WARM_REVISION}",
    rows_sha256=rows_fingerprint([train_ds.column(c) for c in ("image_sha1", "source", "task", "question", "answer")]))
_order = lock_session_plan(CKPT_REPO, _plan, _order, has_checkpoint=resume_dir is not None)   # first launch stores it; later sessions must match it
print(describe_session(len(train_ds), MAX_STEPS, _a.per_device_train_batch_size, _a.gradient_accumulation_steps, _resume_step, ORDER_SEED))
_gate = ResumeGateTrainerCallback()
trainer.add_callback(_gate)                                                      # step / LR / optimizer state / rows left are checked before the first step
trainer.add_callback(TimeBudgetTrainerCallback(budget_s=TIME_BUDGET_H * 3600))   # clean save + push before Colab's 24 h cap
_final_step = train_with_exact_resume(trainer, train_ds, _order, resume_dir, _gate, max_retries=MAX_INSESSION_RETRIES)
ensure_hub_checkpoint(CKPT_REPO, OUT_DIR, _final_step)                           # the session ends only once the Hub holds this step
print(f"session ended at step {_final_step} of {MAX_STEPS}" + ("" if _final_step >= MAX_STEPS else ": run the notebook again to continue from exactly here"))


In [ ]:
# Cell 7 — export merged (policy-carrying) model, gated per the KNOB
MERGED_REPO = "isaacmg/qwen3-vl-8b-hebrew-v23a-merged"
if TRAIN_VISION_LORA:
    _nz = [n for n, p in model.named_parameters()
           if ".visual.blocks." in n and "lora_B" in n and p.detach().abs().max().item() > 0]
    assert len(_nz) == 108, f"shipped adapter tower lora_B nonzero: {len(_nz)}/108"
if MERGER_MODE == "lora":
    _mnz = [n for n, p in model.named_parameters()
            if "merger" in n and "lora_B" in n and p.detach().abs().max().item() > 0]
    assert len(_mnz) == 8, f"merger lora_B nonzero: {len(_mnz)}/8 — merger never moved"
elif MERGER_MODE == "full":
    _moved = []
    for _n, _m in model.named_modules():
        if any(_n.endswith(s) for s in MERGER_MODULES) and hasattr(_m, "modules_to_save"):
            _c = _m.modules_to_save["default"].weight.detach().float()
            _o = _m.original_module.weight.detach().float()
            if not torch.allclose(_c, _o):
                _moved.append(_n)
    assert len(_moved) == 8, f"merger full weights moved: {len(_moved)}/8"
print(f"shipped checks OK (MERGER_MODE={MERGER_MODE})")
model.save_pretrained_merged("v23a-merged", tokenizer, save_method="merged_16bit")
# ship the TRAINING resolution policy with the model (hard rule)
tokenizer.image_processor.save_pretrained("v23a-merged")
model.push_to_hub_merged(MERGED_REPO, tokenizer, save_method="merged_16bit", private=True)
print("pushed", MERGED_REPO)
